# CircuitGraph Quickstart

**Audience:** Python users who want to build circuits programmatically and inspect results as Pandas and NetworkX objects.

This notebook builds a voltage divider with `CircuitGraph`, compiles it to a Xyce netlist, and optionally runs an operating-point analysis when Xyce is installed.

In [ ]:
from pathlib import Path
import shutil
from tempfile import TemporaryDirectory

from xyce_py import CircuitGraph, Resistor, VoltageSource, find_xyce_executable


def build_voltage_divider(*, xyce_path: str = "Xyce", base_out_dir: str = "_xyce_runs") -> CircuitGraph:
    divider = CircuitGraph(xyce_path=xyce_path, base_out_dir=base_out_dir)
    divider.add_node("gnd", is_ground=True)
    divider.add_branch("vin", "gnd", [VoltageSource("supply", 10.0)])
    divider.add_branch("vin", "vout", [Resistor("top", 1000)])
    divider.add_branch("vout", "gnd", [Resistor("bottom", 1000)])
    return divider


def xyce_available() -> bool:
    default_path = Path("/usr/local/XyceNF_7.10/bin/Xyce")
    return default_path.exists() or shutil.which("Xyce") is not None

## Build the topology

`CircuitGraph` is the public topology input. It owns an internal `networkx.MultiDiGraph` because circuits need parallel branches and directed terminal polarity.

In [ ]:
divider_graph = build_voltage_divider()

assert divider_graph.G.nodes["gnd"]["is_ground"] is True
assert divider_graph.G.number_of_edges() == 3

## Compile without running Xyce

Compile-only workflows are useful for reviewing the generated netlist body before a simulation run.

In [ ]:
compiled_body = divider_graph.compile_body()
netlist_preview = "\n".join([*compiled_body.lines, ".END"]) + "\n"
print(netlist_preview)

## Optional operating-point run

The result keeps the waveform table as the canonical numeric output. `solved_graph()` returns a copied topology annotated with selected node voltages.

In [ ]:
if xyce_available():
    with TemporaryDirectory() as tmpdir:
        simulation_graph = build_voltage_divider(
            xyce_path=find_xyce_executable(),
            base_out_dir=tmpdir,
        )

        op_result = simulation_graph.simulate_op()
        print(op_result.translated_waveforms())

        solved_topology = op_result.solved_graph(row=0)
        print("vout solved voltage:", solved_topology.nodes["vout"]["solved_voltage"])
else:
    print("Xyce was not found; the compile-only cells above are still runnable.")